# Customer Risk Analysis

**Case:** CR – Customer Risk Analysis

**Your Mission:** Identify WHO is driving defaults at EduFin and convert risk insights into business action.

---

**Business Context:**  
Portfolio risk was identified in Case 1. Now the focus shifts to customer behavior, risk segments, and decision-making.

**Your Assignment:** Customer risk investigation using SQL analysis.

---

# Dataset Progression

### CR – Customer Risk Analysis

### Step 1: Testing Phase

Dataset used:

`workspace.edufin_small`

Purpose:

* Validate SQL logic
* Test joins, filters, aggregations, and calculations
* Debug queries before scaling

---

### Step 2: Final Production Run

Dataset used:

`workspace.edufin_national`

Purpose:

* Generate final outputs
* Validate results at scale
* Observe risk pattern changes across datasets

---

### Submission Workflow

1. Build and test queries on `edufin_small`
2. Validate logic and outputs
3. Execute final run on `edufin_national`
4. Submit production-level results only

---

# Deliverables

- Queries 2A–2E
- Executive Summary
- WHY Gate answers
- Scale comparison observations

---

# BEFORE YOU START

1. Run Data Validation Check (Cell 2)
2. Solve queries sequentially
3. Follow dataset workflow before submission
4. Compare testing vs production outputs

In [0]:
%sql
-- DATA VALIDATION CHECK
-- Run this cell FIRST to verify your environment is ready
-- This is NOT a graded query - just a system check

SELECT 
  COUNT(*) AS total_loans,
  COUNT(DISTINCT customer_id) AS unique_customers,
  COUNT(DISTINCT l.customer_id) AS customers_with_loans,
  MIN(loan_amount) AS min_loan,
  MAX(loan_amount) AS max_loan,
  COUNT(DISTINCT CASE WHEN loan_status = 'Defaulted' THEN customer_id END) AS customers_with_defaults
FROM workspace.edufin_small.loans_l;

-- Expected: ~5000 loans, ~2400 unique customers, some customers with defaults
-- If you see 0 loans or errors, contact support before proceeding

total_loans,unique_customers,customers_with_loans,min_loan,max_loan,customers_with_defaults
5000,2432,2432,-1142532.96,1497594.15,597


In [0]:
%sql
-- QUERY 2A: Customer Credit Risk Segmentation (CIBIL-Based)
--
-- BUSINESS PURPOSE:
-- Understand how customer credit quality (CIBIL score) is related to loan default behavior.
-- This analysis is used to identify risk patterns across different credit segments and support lending decisions.
--
-- CIBIL Segments:
-- - <600        → Very High Risk
-- - 600–650     → High Risk
-- - 651–700     → Medium Risk
-- - 701–750     → Low Risk
-- - 750+        → Very Low Risk
--
-- WHAT YOU NEED TO DO:
-- 1. Group customers into the above CIBIL segments
-- 2. Analyze default behavior across these segments
-- 3. Compare risk levels across segments
--
-- EXPECTED ANALYSIS:
-- - Identify which segments show higher risk behavior
-- - Observe whether risk reduces as CIBIL score improves
-- - Derive insights for lending strategy and risk control
--
-- NOTE:
-- You will need to decide the appropriate aggregation logic to represent “default behavior” meaningfully.
--
-- Write your query below:

SELECT
    CASE
        WHEN c.cibil_score < 600 THEN 'Very High Risk (<600)'
        WHEN c.cibil_score BETWEEN 600 AND 650 THEN 'High Risk (600-650)'
        WHEN c.cibil_score BETWEEN 651 AND 700 THEN 'Medium Risk (651-700)'
        WHEN c.cibil_score BETWEEN 701 AND 750 THEN 'Low Risk (701-750)'
        ELSE 'Very Low Risk (750+)'
    END AS cibil_segment,

    COUNT(DISTINCT l.customer_id) AS total_customers,
    COUNT(*) AS total_loans,

    COUNT(CASE WHEN l.loan_status = 'Defaulted' THEN 1 END) AS defaulted_loans,
    COUNT(DISTINCT CASE WHEN l.loan_status = 'Defaulted' THEN l.customer_id END) AS customers_with_defaults,

    ROUND(
        100.0 * COUNT(CASE WHEN l.loan_status = 'Defaulted' THEN 1 END) / COUNT(*),
        2
    ) AS default_rate_percent

FROM workspace.edufin_small.loans l
INNER JOIN workspace.edufin_small.customers c ON l.customer_id = c.customer_id
GROUP BY
    CASE
        WHEN c.cibil_score < 600 THEN 'Very High Risk (<600)'
        WHEN c.cibil_score BETWEEN 600 AND 650 THEN 'High Risk (600-650)'
        WHEN c.cibil_score BETWEEN 651 AND 700 THEN 'Medium Risk (651-700)'
        WHEN c.cibil_score BETWEEN 701 AND 750 THEN 'Low Risk (701-750)'
        ELSE 'Very Low Risk (750+)'
    END
ORDER BY
    MIN(c.cibil_score);

cibil_segment,total_customers,total_loans,defaulted_loans,customers_with_defaults,default_rate_percent
Very High Risk (<600),1349,2800,432,382,15.43
High Risk (600-650),314,640,71,64,11.09
Medium Risk (651-700),262,527,51,47,9.68
Low Risk (701-750),297,607,67,62,11.04
Very Low Risk (750+),210,426,49,42,11.50


In [0]:
%sql
SELECT
    CASE
        WHEN c.cibil_score < 600 THEN 'Very High Risk (<600)'
        WHEN c.cibil_score BETWEEN 600 AND 650 THEN 'High Risk (600-650)'
        WHEN c.cibil_score BETWEEN 651 AND 700 THEN 'Medium Risk (651-700)'
        WHEN c.cibil_score BETWEEN 701 AND 750 THEN 'Low Risk (701-750)'
        ELSE 'Very Low Risk (750+)'
    END AS cibil_segment,

    COUNT(DISTINCT l.customer_id) AS total_customers,
    COUNT(*) AS total_loans,

    COUNT(CASE WHEN l.loan_status = 'Defaulted' THEN 1 END) AS defaulted_loans,
    COUNT(DISTINCT CASE WHEN l.loan_status = 'Defaulted' THEN l.customer_id END) AS customers_with_defaults,

    ROUND(
        100.0 * COUNT(CASE WHEN l.loan_status = 'Defaulted' THEN 1 END) / COUNT(*),
        2
    ) AS default_rate_percent

FROM workspace.edufin_small.loans l
INNER JOIN workspace.edufin_small.customers c ON l.customer_id = c.customer_id
GROUP BY
    CASE
        WHEN c.cibil_score < 600 THEN 'Very High Risk (<600)'
        WHEN c.cibil_score BETWEEN 600 AND 650 THEN 'High Risk (600-650)'
        WHEN c.cibil_score BETWEEN 651 AND 700 THEN 'Medium Risk (651-700)'
        WHEN c.cibil_score BETWEEN 701 AND 750 THEN 'Low Risk (701-750)'
        ELSE 'Very Low Risk (750+)'
    END
ORDER BY
    MIN(c.cibil_score);

cibil_segment,total_customers,total_loans,defaulted_loans,customers_with_defaults,default_rate_percent
Very High Risk (<600),1349,2800,432,382,15.43
High Risk (600-650),314,640,71,64,11.09
Medium Risk (651-700),262,527,51,47,9.68
Low Risk (701-750),297,607,67,62,11.04
Very Low Risk (750+),210,426,49,42,11.50


In [0]:
%sql
-- QUERY 2B: Customer Risk Segmentation (Age-Based)
--
-- BUSINESS PURPOSE:
-- Understand how customer age influences loan default behavior and risk exposure.
-- Instead of analyzing individual borrowers, customers are grouped into age segments
-- to identify risk patterns across different life stages.
--
-- This helps in:
-- - Understanding risk variation across customer age groups
-- - Identifying whether specific age brackets show higher default tendencies
-- - Supporting age-based underwriting and lending strategy decisions
--
-- AGE SEGMENTS:
-- - 18–25
-- - 26–30
-- - 31–35
--
-- WHAT YOU NEED TO DO:
-- 1. Group customers into the defined age segments
-- 2. Compare risk behavior across these groups
-- 3. Analyze loan behavior differences across age groups
--
-- IMPORTANT CONDITIONS:
-- - Consider only customers who have taken loans
-- - Ensure segments are meaningfully comparable (avoid interpreting unstable or low-volume groups directly)
--
-- EXPECTED ANALYSIS:
-- - Identify which age group shows higher risk behavior
-- - Observe whether risk varies across age brackets
-- - Understand how borrowing patterns differ by age
--
-- NOTE:
-- You will need to decide the appropriate logic to measure and compare “risk behavior” across age groups.
--
-- Write your query below:

In [0]:
%sql
SELECT
    CASE
        WHEN c.age BETWEEN 18 AND 25 THEN '18-25'
        WHEN c.age BETWEEN 26 AND 30 THEN '26-30'
        WHEN c.age BETWEEN 31 AND 35 THEN '31-35'
        ELSE 'Other'
    END AS age_segment,

    COUNT(DISTINCT l.customer_id) AS total_customers,
    COUNT(*) AS total_loans,

    COUNT(CASE WHEN l.loan_status = 'Defaulted' THEN 1 END) AS defaulted_loans,
    COUNT(DISTINCT CASE WHEN l.loan_status = 'Defaulted' THEN l.customer_id END) AS customers_with_defaults,

    ROUND(
        100.0 * COUNT(CASE WHEN l.loan_status = 'Defaulted' THEN 1 END) / COUNT(*),
        2
    ) AS default_rate_percent,

    ROUND(AVG(l.loan_amount), 2) AS avg_loan_amount,
    ROUND(AVG(c.cibil_score), 2) AS avg_cibil_score

FROM workspace.edufin_small.loans l
INNER JOIN workspace.edufin_small.customers c ON l.customer_id = c.customer_id
WHERE c.age BETWEEN 18 AND 35
GROUP BY
    CASE
        WHEN c.age BETWEEN 18 AND 25 THEN '18-25'
        WHEN c.age BETWEEN 26 AND 30 THEN '26-30'
        WHEN c.age BETWEEN 31 AND 35 THEN '31-35'
        ELSE 'Other'
    END
ORDER BY
    MIN(c.age);

age_segment,total_customers,total_loans,defaulted_loans,customers_with_defaults,default_rate_percent,avg_loan_amount,avg_cibil_score
18-25,1605,3281,447,402,13.62,405860.41,584.41
26-30,502,1059,141,121,13.31,422643.8,588.17
31-35,325,660,82,74,12.42,422386.38,600.2


In [0]:
%sql
-- QUERY 2C: Income-Based Default Loss Analysis
--
-- BUSINESS PURPOSE:
-- Understand how customer income levels impact financial loss due to loan defaults.
-- Instead of only looking at default rates, this analysis focuses on identifying
-- which income segments are responsible for the highest monetary losses.
--
-- This helps in:
-- - Identifying high financial loss segments
-- - Understanding which income groups contribute most to credit losses
-- - Supporting pricing, lending limits, and risk-based policy decisions
--
-- INCOME SEGMENTS:
-- - <3L
-- - 3L–6L
-- - 6L–12L
-- - 12L+
--
-- WHAT YOU NEED TO DO:
-- 1. Group customers into the defined income segments
-- 2. Analyze default behavior across segments
-- 3. Measure financial impact of defaults in each segment
--
-- IMPORTANT CONDITIONS:
-- - Consider only customers who have taken loans
-- - Focus on identifying segments with higher financial exposure, not just frequency
--
-- EXPECTED ANALYSIS:
-- - Identify which income segment contributes the highest total default loss
-- - Observe whether lower income groups show higher default frequency or loss concentration
-- - Understand how financial risk is distributed across income levels
--
-- NOTE:
-- You will need to decide how to meaningfully represent “default loss” at segment level.
--
-- Write your query below:


In [0]:
%sql
SELECT
    CASE
        WHEN c.annual_income < 300000 THEN '<3L'
        WHEN c.annual_income BETWEEN 300000 AND 599999 THEN '3L-6L'
        WHEN c.annual_income BETWEEN 600000 AND 1199999 THEN '6L-12L'
        WHEN c.annual_income >= 1200000 THEN '12L+'
        ELSE 'Unknown'
    END AS income_segment,

    COUNT(DISTINCT l.customer_id) AS total_customers,
    COUNT(*) AS total_loans,

    COUNT(CASE WHEN l.loan_status = 'Defaulted' THEN 1 END) AS defaulted_loans,
    COUNT(DISTINCT CASE WHEN l.loan_status = 'Defaulted' THEN l.customer_id END) AS customers_with_defaults,

    ROUND(
        100.0 * COUNT(CASE WHEN l.loan_status = 'Defaulted' THEN 1 END) / COUNT(*),
        2
    ) AS default_rate_percent,

    ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN l.loan_amount ELSE 0 END), 2) AS total_default_loss,
    ROUND(AVG(CASE WHEN l.loan_status = 'Defaulted' THEN l.loan_amount END), 2) AS avg_default_loss_per_loan,
    ROUND(AVG(l.loan_amount), 2) AS avg_loan_amount,
    ROUND(AVG(c.cibil_score), 2) AS avg_cibil_score

FROM workspace.edufin_small.loans l
INNER JOIN workspace.edufin_small.customers c ON l.customer_id = c.customer_id
GROUP BY
    CASE
        WHEN c.annual_income < 300000 THEN '<3L'
        WHEN c.annual_income BETWEEN 300000 AND 599999 THEN '3L-6L'
        WHEN c.annual_income BETWEEN 600000 AND 1199999 THEN '6L-12L'
        WHEN c.annual_income >= 1200000 THEN '12L+'
        ELSE 'Unknown'
    END
ORDER BY
    MIN(c.annual_income);

income_segment,total_customers,total_loans,defaulted_loans,customers_with_defaults,default_rate_percent,total_default_loss,avg_default_loss_per_loan,avg_loan_amount,avg_cibil_score
<3L,1003,2068,341,297,16.49,5.526310541E7,162061.89,162993.73,482.37
3L-6L,536,1093,115,106,10.52,4.398559782E7,382483.46,417115.46,623.16
6L-12L,524,1074,142,127,13.22,7.362423491E7,518480.53,518330.32,650.33
12L+,369,765,72,67,9.41,6.446118616E7,895294.25,925905.66,731.17


In [0]:
%sql
-- QUERY 2D: Multi-Dimensional Risk Segmentation (Employment + Income)
--
-- BUSINESS PURPOSE:
-- Understand how the combination of employment type and income level influences loan default risk.
-- Instead of analyzing each factor separately, this approach identifies high-risk borrower profiles
-- by combining financial capacity (income) with job stability (employment type).
--
-- This helps in:
-- - Designing more accurate underwriting rules
-- - Identifying risky borrower personas (e.g., low-income self-employed customers)
-- - Improving risk-based segmentation for lending and recovery strategies
--
-- INCOME SEGMENTS:
-- - <3L
-- - 3L–6L
-- - 6L–12L
-- - 12L+
--
-- WHAT YOU NEED TO DO:
-- 1. Create combined segments using employment type + income bracket
-- 2. Analyze default behavior across these combined segments
-- 3. Evaluate overall portfolio exposure for each segment
--
-- IMPORTANT CONDITIONS:
-- - Exclude customers with missing employment type
-- - Ensure segments with very low data points are not interpreted in isolation
--
-- EXPECTED ANALYSIS:
-- - Identify which employment + income combinations show highest default risk
-- - Understand whether self-employed customers are consistently riskier across income levels
-- - Observe how job stability impacts risk within the same income bracket
--
-- NOTE:
-- You will need to decide the best way to represent combined “risk behavior” at segment level.
--
-- Write your query below:


In [0]:
%sql
SELECT
    c.employment_type,
    CASE
        WHEN c.annual_income < 300000 THEN '<3L'
        WHEN c.annual_income BETWEEN 300000 AND 599999 THEN '3L-6L'
        WHEN c.annual_income BETWEEN 600000 AND 1199999 THEN '6L-12L'
        WHEN c.annual_income >= 1200000 THEN '12L+'
        ELSE 'Unknown'
    END AS income_segment,

    COUNT(DISTINCT l.customer_id) AS total_customers,
    COUNT(*) AS total_loans,

    COUNT(CASE WHEN l.loan_status = 'Defaulted' THEN 1 END) AS defaulted_loans,
    COUNT(DISTINCT CASE WHEN l.loan_status = 'Defaulted' THEN l.customer_id END) AS customers_with_defaults,

    ROUND(
        100.0 * COUNT(CASE WHEN l.loan_status = 'Defaulted' THEN 1 END) / COUNT(*),
        2
    ) AS default_rate_percent,

    ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN l.loan_amount ELSE 0 END), 2) AS total_default_loss,
    ROUND(AVG(l.loan_amount), 2) AS avg_loan_amount,
    ROUND(AVG(c.cibil_score), 2) AS avg_cibil_score

FROM workspace.edufin_small.loans l
INNER JOIN workspace.edufin_small.customers c ON l.customer_id = c.customer_id
WHERE c.employment_type IS NOT NULL
GROUP BY
    c.employment_type,
    CASE
        WHEN c.annual_income < 300000 THEN '<3L'
        WHEN c.annual_income BETWEEN 300000 AND 599999 THEN '3L-6L'
        WHEN c.annual_income BETWEEN 600000 AND 1199999 THEN '6L-12L'
        WHEN c.annual_income >= 1200000 THEN '12L+'
        ELSE 'Unknown'
    END
ORDER BY
    c.employment_type,
    MIN(c.annual_income);

employment_type,income_segment,total_customers,total_loans,defaulted_loans,customers_with_defaults,default_rate_percent,total_default_loss,avg_loan_amount,avg_cibil_score
Salaried,<3L,278,580,101,91,17.41,1.516748339E7,164947.36,483.82
Salaried,3L-6L,165,351,36,32,10.26,1.29557132E7,406137.82,614.16
Salaried,6L-12L,143,284,34,31,11.97,1.734587906E7,531380.9,655.85
Salaried,12L+,99,210,15,15,7.14,1.60817065E7,932106.25,738.57
Self-Employed,<3L,144,303,48,42,15.84,8528487.06,164377.99,483.63
Self-Employed,3L-6L,83,177,20,19,11.30,7838173.55,446853.19,626.99
Self-Employed,6L-12L,72,156,25,23,16.03,1.337711087E7,473410.21,643.1
Self-Employed,12L+,57,113,12,11,10.62,9769862.28,945649.71,737.47
Student,<3L,425,869,140,120,16.11,2.314901689E7,162457.54,484.49
Student,3L-6L,212,410,44,41,10.73,1.720962378E7,410390.94,621.77


In [0]:
%sql
-- QUERY 2E: Customer Risk Prioritization for Collections
--
-- BUSINESS PURPOSE:
-- Identify and prioritize high-risk customers for collections teams using multiple risk indicators.
-- Instead of simple segmentation, this analysis builds a ranked risk view to support recovery actions
-- and minimize financial losses from defaults.
--
-- This helps in:
-- - Prioritizing collection efforts based on risk severity
-- - Reducing loss exposure through targeted recovery actions
-- - Improving operational efficiency of collections teams
--
-- WHAT YOU NEED TO DO:
-- 1. Identify customers with default history
-- 2. Measure their risk exposure using key indicators (default amount, frequency, credit behavior)
-- 3. Create a ranking to prioritize customers from highest to lowest risk
--
-- FINAL OUTPUT SHOULD INCLUDE:
-- - Customer identification details
-- - Measures of default exposure
-- - Risk category (High / Medium / Low or equivalent logic)
-- - Priority ranking for collections action
--
-- IMPORTANT CONDITIONS:
-- - Consider only customers with at least one default event
-- - Focus on relative prioritization rather than isolated metrics
--
-- EXPECTED ANALYSIS:
-- - Identify top priority customers for immediate recovery action
-- - Understand which customers contribute most to portfolio risk
-- - Support efficient allocation of collections resources
--
-- NOTE:
-- You will need to define a meaningful way to combine multiple risk indicators into a single priority ranking.
--
-- Write your query below:

In [0]:
%sql
WITH customer_defaults AS (
    SELECT
        l.customer_id,
        c.full_name,
        c.cibil_score,
        c.annual_income,
        c.employment_type,
        c.age,
        
        -- Default exposure metrics
        COUNT(*) AS total_defaults,
        ROUND(SUM(l.loan_amount), 2) AS total_default_amount,
        ROUND(AVG(l.loan_amount), 2) AS avg_default_amount,
        
        -- Calculate priority score (weighted combination of risk factors)
        -- Higher score = Higher priority for collections
        ROUND(
            (SUM(l.loan_amount) / 100000) * 0.5 +  -- Default amount weight (50%)
            COUNT(*) * 0.3 +                        -- Frequency weight (30%)
            ((900 - c.cibil_score) / 100) * 0.2,    -- Credit quality weight (20%)
            2
        ) AS priority_score
        
    FROM workspace.edufin_small.loans l
    INNER JOIN workspace.edufin_small.customers c ON l.customer_id = c.customer_id
    WHERE l.loan_status = 'Defaulted'
    GROUP BY 
        l.customer_id,
        c.full_name,
        c.cibil_score,
        c.annual_income,
        c.employment_type,
        c.age
)

SELECT
    customer_id,
    full_name,
    
    -- Risk exposure metrics
    total_defaults,
    total_default_amount,
    avg_default_amount,
    
    -- Customer profile
    cibil_score,
    annual_income,
    age,
    employment_type,
    
    -- Priority score and ranking
    priority_score,
    ROW_NUMBER() OVER (ORDER BY priority_score DESC) AS priority_rank,
    
    -- Risk category
    CASE
        WHEN priority_score >= 15 THEN 'High Risk'
        WHEN priority_score >= 8 THEN 'Medium Risk'
        ELSE 'Low Risk'
    END AS risk_category
    
FROM customer_defaults
ORDER BY priority_score DESC, total_default_amount DESC
LIMIT 100;

customer_id,full_name,total_defaults,total_default_amount,avg_default_amount,cibil_score,annual_income,age,employment_type,priority_score,priority_rank,risk_category
554,Chakrika Tata,2,2644143.36,1322071.68,766,2117279.72,24,Student,14.09,1,Medium Risk
2122,Yutika Pandey,2,2422868.04,1211434.02,830,1728509.55,21,Unemployed,12.85,2,Medium Risk
2363,Ridhi Venkataraman,2,2302174.95,1151087.48,849,1368669.95,27,Unemployed,12.21,3,Medium Risk
1155,Reyansh Gupta,2,2079994.64,1039997.32,843,1177783.47,19,Student,11.11,4,Medium Risk
176,Vasana Bobal,2,2033636.4,1016818.2,765,943359.52,21,Student,11.04,5,Medium Risk
1599,Suhani Kara,4,1510620.3,377655.08,679,735362.66,27,Salaried,9.2,6,Medium Risk
712,Kashish Manne,1,1484912.05,1484912.05,624,1987619.0,32,Salaried,8.28,7,Medium Risk
2378,Christopher Sharaf,1,1441545.21,1441545.21,616,2071213.5,23,Student,8.08,8,Medium Risk
2661,Vamakshi Verma,2,1463272.94,731636.47,836,1712940.92,22,null,8.04,9,Medium Risk
385,Amruta Dhar,1,1424094.12,1424094.12,624,944334.11,32,Salaried,7.97,10,Low Risk


## Executive Summary (Case 2: Customer Risk & Collections Strategy)

**Based on your analysis from Query 2A to 2E, answer the following:**

---

### **1. Credit Risk Pattern (CIBIL Analysis - Query 2A)**

What trend do you observe between CIBIL score segments and default rates?  
Is there a clear relationship between lower credit scores and higher default behavior?

**Analytical Findings:** _Document your interpretation and business insight below_

**Observation:**

Yes, there is a clear relationship between CIBIL scores and default behavior, though not perfectly linear. The <600 segment shows the highest default rate at 15.43%, which is significantly higher than other segments. This confirms that customers with very low credit scores are the riskiest.

However, an interesting pattern emerges in the mid-to-high ranges. The 651-700 segment actually has the lowest default rate at 9.68%, which is lower than even the 750+ segment (11.50%). This suggests that the relationship is not simply "higher CIBIL = lower risk" across all ranges.

The 600-650 and 701-750 segments both show similar default rates around 11%, indicating that once customers cross the 600 threshold, the risk becomes more stable. The slightly elevated risk in the 750+ segment might be explained by larger loan amounts or different borrower behavior among high-credit customers.

**Key Takeaway:** While lower CIBIL scores (<600) clearly indicate higher default risk, the mid-range segments (651-750) appear to be the most stable from a risk perspective. This finding should influence our underwriting strategy - we may want to be particularly cautious with sub-600 applicants while treating the 650-750 range as a sweet spot for lending.

### **2. Demographic Risk Insight (Age Analysis - Query 2B)**

Which age group shows the highest default rate?  
Do younger borrowers exhibit higher risk compared to mid-career or older customers?

**Analytical Findings:** _Document your interpretation and business insight below_

**Observation:**

Yes, younger borrowers clearly show higher risk. The 18-25 age group has the highest default rate at 13.62%, followed by the 26-30 group at 13.31%, and the 31-35 group at 12.42%. This shows a steady decline in risk as customers get older.

The difference might seem small (only about 1.2% between youngest and oldest), but when you look at the actual numbers, it's significant. The 18-25 group has 1,605 customers with 3,281 loans, making them the largest segment. So even a slightly higher default rate means more actual losses.

Why does this happen? Younger customers are probably less financially stable - they're just starting their careers, their income might not be steady yet, and they have less experience managing money and loans. As people move into their late 20s and early 30s, they typically have more stable jobs, better income, and more financial maturity.

**Key Takeaway:** Age matters for loan risk. The 18-25 group needs extra attention because they have both the highest default rate and the largest customer base. We might want to be more careful when approving loans for very young borrowers, or maybe offer them smaller loan amounts until they build a better repayment track record.


### **3. Income Risk & Loss Concentration (Query 2C)**

Which income segment contributes the highest total defaulted amount?  
Is risk driven more by low-income borrowers (higher default rate) or high-income borrowers (larger loan sizes)?


**Analytical Findings:** _Document your interpretation and business insight below_

**Observation:**

The 6L-12L income segment contributes the highest total defaulted amount at ₹73.6 million, even though it doesn't have the highest default rate. This is interesting because the <3L segment has a higher default rate (16.49%) but only ₹55.3 million in losses. So the answer is: risk is driven by BOTH factors, but in different ways.

Low-income borrowers (<3L) default more often - they have the highest percentage of defaults. But mid-income borrowers (6L-12L) create the biggest financial losses because they take out larger loans. When they default, each loss is much bigger (around ₹5.2 lakhs per default compared to ₹1.6 lakhs for low-income borrowers).

The high-income segment (12L+) is interesting too. They have the lowest default rate at only 9.41%, but they still contribute ₹64.5 million in total losses because when high-income borrowers default, the amounts are huge (almost ₹9 lakhs per default on average).

**Key Takeaway:** We need to watch both types of risk. Low-income borrowers need attention because many of them default (high frequency risk). But mid-to-high income borrowers need attention too because when they default, we lose much more money per case (high severity risk). The 6L-12L segment is our biggest problem right now because it combines both - decent number of defaults AND large loan amounts.

### **4. High-Risk Customer Profile (Query 2D)**

Based on combined segmentation (employment type + income),  
which customer profile has the highest default rate?

Should EduFin tighten underwriting criteria for this specific segment?

**Analytical Findings:** _Document your interpretation and business insight below_

**Observation:**

The highest default rate is 17.41%, and it belongs to Salaried employees earning less than 3L. This is followed closely by Students (<3L) at 16.11% and Unemployed (<3L) at 16.17%. So basically, anyone earning under 3 lakhs has high risk, no matter what their job type is.

But here's something surprising: Self-employed people in the 6L-12L range have a 16.03% default rate, which is almost as high as the low-income groups. This is unusual because they're earning decent money but still defaulting a lot. This might mean their income is not stable - maybe their business has ups and downs.

Yes, EduFin should definitely tighten rules for these segments. The <3L group across all employment types is clearly risky. Also, we need to look more carefully at self-employed mid-income borrowers (6L-12L) because they're showing unexpected risk despite having better income.

**Key Takeaway:** Income matters more than job type. All <3L borrowers are high risk regardless of whether they're salaried, students, or unemployed. But we also found a hidden risk group - self-employed people earning 6L-12L need extra checking because their income might look good on paper but isn't stable enough.

### **5. Collections Strategy Effectiveness (Priority Model - Query 2E)**

Looking at the top-ranked customers by priority score:
- Are they driven more by high default amounts, low CIBIL scores, or repeated defaults?
- Does the prioritization model successfully identify high-impact recovery targets?

**Analytical Findings:** _Document your interpretation and business insight below_

**Observation:**

Looking at the top customers, they're mainly driven by high default amounts. The #1 priority customer (Chakrika Tata) has 2 defaults totaling ₹26.4 lakhs, and most top 10 customers have similar patterns - large loan amounts that defaulted. The priority score gives 50% weight to the money owed, so big defaults naturally rank higher.

Interestingly, many top-priority customers have decent CIBIL scores (between 600-850). This tells us they're not career defaulters - something went wrong recently. This is actually good news for collections because these people might be easier to recover from compared to people who always had bad credit.

Yes, the model works well. It successfully identifies high-impact targets - people who owe a lot of money but might still be reachable. For example, customer #6 (Suhani Kara) has 4 defaults but smaller amounts, while customer #1 has only 2 defaults but huge amounts. The model correctly puts #1 higher because recovering that money has bigger impact.

**Key Takeaway:** The priority model is doing its job right. It focuses the collections team on customers who owe the most money AND have some chance of paying back (based on their CIBIL scores). The top 20-30 customers should get immediate attention because they represent the biggest recovery opportunity - large amounts owed by people who aren't hopeless cases.

### **6. Ideal High-Risk Borrower Profile (Final Synthesis)**

Based on all analyses (CIBIL, Age, Income, Employment, Behavior),  
describe the "typical high-risk borrower" EduFin should monitor closely.

Include:
- CIBIL range
- Age group
- Income bracket
- Employment type
- Loan/default behavior

**Analytical Findings:** _Document your interpretation and business insight below_

**Observation:**

Based on all our analysis, here's the profile of a typical high-risk borrower that EduFin should watch closely:

**CIBIL Score:** Below 600 (very high risk), but also watch the 600-650 range carefully.

**Age Group:** 18-25 years old. These young borrowers show the highest default rates and represent the largest group, so even small increases in their risk create big losses.

**Income Bracket:** Less than 3 lakhs annually. This is the biggest red flag - low income leads to high default rates across ALL job types. The 6L-12L range also needs attention because they create the largest total losses.

**Employment Type:** Any employment type earning <3L is risky, but pay special attention to Self-employed people in the 6L-12L range (they have unexpectedly high default rates of 16% despite decent income).

**Loan/Default Behavior:** Multiple small loans or one large loan. Watch for customers who keep coming back for more loans without fully paying off previous ones. Also monitor customers with loan amounts that seem too big compared to their income.

**Key Takeaway:** The highest-risk profile is: Young (18-25), Low CIBIL (<600), Low Income (<3L), Any employment type. This combination shows up repeatedly across all our analyses. EduFin should either reject these applications or offer much smaller loan amounts with stricter monitoring.

## INVESTIGATION SUMMARY

**Write a 3–4 sentence executive summary synthesizing your complete customer risk analysis:**

---

**My Summary:**

Our analysis shows that low-income borrowers earning less than 3 lakhs have the highest default rates (16.49%) across all employment types, with young borrowers aged 18-25 adding extra risk at 13.62% default rate. The <600 CIBIL segment clearly indicates higher risk (15.43% default rate), but we found that the biggest financial losses come from the 6L-12L income segment (₹73.6 million total) because they take larger loans - showing that risk has both frequency and severity components. Self-employed borrowers in the 6L-12L range are a hidden risk group with unexpectedly high 16.03% default rates despite decent income. Based on these findings, EduFin should tighten underwriting for young (<25), low-income (<3L), and sub-600 CIBIL applicants while prioritizing collections efforts on the top 20-30 high-value defaulters who represent the biggest recovery opportunity.

%md
## WHY GATE - Business Reasoning

### Question 1: Aggregation Logic (Customer vs Loan Level)

Across your analysis (especially Query 2A–2E), you use both COUNT(DISTINCT customer_id) and COUNT(loan_id).  
Why is it important to distinguish between customer-level and loan-level aggregation?  
How does using COUNT(loan_id) ensure accurate calculation of default rates?

**Analytical Findings:** _Document your interpretation and business insight below_

**Observation:**

This is important because one customer can have multiple loans. If we only count customers (COUNT DISTINCT customer_id), we might miss the full picture of risk behavior.

For example, imagine Customer A has 1 loan that defaulted, and Customer B has 5 loans where 3 defaulted. If we just count customers, both look the same (1 customer each). But if we count loans, we see the real impact - Customer B is causing 3x more defaults.

Using COUNT(loan_id) for default rate calculation is more accurate because default rate means "what percentage of LOANS default," not "what percentage of CUSTOMERS default." A customer with 5 loans who defaults on 2 of them is different from a customer with 1 loan who defaults on it - both are 1 defaulted customer, but very different loan-level risk.

In our queries, we calculated default rate as: (Number of defaulted loans / Total number of loans) × 100. This gives us the true risk per loan, which is what matters for business - how many loans are at risk, not just how many people.

**Key Takeaway:** Customer-level metrics tell us WHO is risky. Loan-level metrics tell us HOW MUCH risk exists. Both are needed - customer counts help size the problem (how many people), while loan counts help measure the actual financial exposure (how many transactions).


### Question 2: Segment-Based Risk vs Individual Thresholds

In Query 2C, instead of filtering customers using a fixed income cutoff (e.g., income > ₹5L), you analyzed default behavior across income segments.

Why does segment-level analysis provide more reliable risk insights than using arbitrary thresholds?
How can segmentation reveal patterns that fixed cutoffs may hide?


**Analytical Findings:** _Document your interpretation and business insight below_

**Observation:**

Fixed thresholds are too simple and can hide important patterns. If we just said "income > ₹5L = low risk," we'd be treating someone earning ₹5.1L the same as someone earning ₹20L, which doesn't make sense.

Segment-level analysis is better because it shows us HOW risk changes at different income levels. We found that the <3L group has 16.49% default rate, but the 6L-12L group actually creates the biggest losses (₹73.6 million) even though their default rate is lower. A fixed cutoff at ₹5L would have completely missed this pattern.

Here's what segmentation reveals that fixed cutoffs hide:

1. **Non-linear patterns**: Risk doesn't drop smoothly. Our data showed the 12L+ group has lower default rates (9.41%) but still causes big losses because of huge loan amounts.

2. **Sweet spots and danger zones**: We discovered the 6L-12L segment is our biggest problem - not too high, not too low, but the perfect storm of decent volume AND large ticket sizes.

3. **Actionable groups**: Instead of one rule for everyone, we can now create different strategies - strict for <3L, careful monitoring for 6L-12L, different terms for 12L+.

**Key Takeaway:** Segmentation lets us see the full story across the income range. Fixed cutoffs are like looking through a keyhole - you only see one narrow view. Segments are like opening the door - you see the entire room and all the furniture (patterns) inside.


### Question 3: Multi-Factor Risk Modeling (Beyond Single Variable)

Query 2E introduces a priority score combining multiple factors (default amount, CIBIL score, and default count).  
Why is a multi-factor scoring approach more effective than relying on a single metric like default amount alone?


**Analytical Findings:** _Document your interpretation and business insight below_

**Observation:**

Using only one metric can be misleading. If we only looked at default amount, we'd chase after every big loan that defaulted, even if the person has good credit and just had temporary trouble (easier to recover). If we only looked at CIBIL score, we'd waste time on small-amount chronic defaulters.

A multi-factor score is like using multiple clues to solve a mystery instead of just one. Our priority score combined three things:

1. **Default Amount (50% weight)**: How much money is at stake? This matters most for financial impact.

2. **Default Count (30% weight)**: Is this a one-time problem or a pattern? Repeat defaulters need different handling.

3. **CIBIL Score (20% weight)**: What's their credit history? Someone with a 700 CIBIL who just defaulted is different from someone with a 400 CIBIL who defaults repeatedly.

The weights reflect business priorities - recovering money is most important (50%), but we also care about behavior patterns (30%) and creditworthiness (20%).

Real example from our data: Customer A defaulted on ₹26 lakhs (2 loans, decent CIBIL). Customer B defaulted on ₹8 lakhs (5 loans, very low CIBIL). Single-metric thinking would rank A first (biggest amount). But our multi-factor score considers that A might pay back (better credit), while B is a chronic problem (many defaults, bad CIBIL). Both need attention but for different reasons.

**Key Takeaway:** Multi-factor scoring balances different types of risk. It's like a doctor checking blood pressure, heart rate, AND temperature instead of just one - you get a complete health picture, not just one symptom.

### Question 4: Impact of Sample Size on Risk Insights

When comparing results from edufin_small and edufin_national, some default rate patterns may stabilize while others may change.

Why does increasing sample size improve statistical confidence in risk analysis?
What does it mean if a trend observed in a small dataset weakens or disappears in a larger dataset?

**Analytical Findings:** _Document your interpretation and business insight below_

**Observation:**

Larger samples make our findings more reliable because random chance has less impact. Think of it like flipping a coin - if you flip 10 times and get 7 heads, you might think the coin is unfair. But if you flip 1,000 times and get 700 heads, NOW you can be confident something is actually wrong with the coin.

In our analysis, edufin_small has 5,000 loans and 2,432 customers. That's decent, but some segments are really small. For example, if only 50 customers are in the "Self-employed 6L-12L" group, and 8 defaulted, we get 16% default rate. But is that a real pattern or just bad luck with those specific 50 people?

With edufin_national (much bigger dataset), we might have 500 or 5,000 customers in that same segment. If the 16% default rate holds steady, we can be confident it's a REAL pattern. If it drops to 10%, then the small dataset was misleading - we just happened to sample more risky people by chance.

When a trend weakens or disappears in larger data, it usually means one of these things:

1. **It was never real** - just random noise in the small sample (like flipping 7 heads out of 10)
2. **Small sample had bias** - maybe edufin_small only included customers from certain regions or time periods
3. **The effect is real but small** - large data dilutes it, showing it's not as important as we thought

**Key Takeaway:** Bigger samples = more confidence. If we find a pattern in small data, we should always check if it holds in bigger data before making business decisions. Small data can trick us into seeing patterns that don't really exist.

### Question 5: Scale Validation (Small vs National Dataset)

If a relationship exists in edufin_small but disappears in edufin_national, what could that imply about sample bias, statistical stability, and data representativeness?

**Analytical Findings:** _Document your interpretation and business insight below_

**Observation:**

If a relationship exists in edufin_small but disappears in edufin_national, it raises three important questions:

**1. Sample Bias:** The small dataset might not represent the full population fairly. For example, maybe edufin_small only has customers from urban areas, or only from certain months, or only from specific branches. If self-employed 6L-12L customers show 16% default rate in small data but 11% in national data, maybe the small sample accidentally included more risky self-employed people.

**2. Statistical Stability:** Small samples have more "noise" - random ups and downs that don't mean anything. Imagine looking at only 5 days of weather vs. looking at the whole year. One week might seem unusually rainy, but the yearly average tells the real story. If a pattern disappears in bigger data, it might have been just random noise all along.

**3. Data Representativeness:** The small dataset might be from a specific time period or situation. For example, if edufin_small was collected during an economic crisis, default rates would be higher everywhere. When we look at edufin_national (which includes normal times too), the pattern looks different because it represents the full reality, not just crisis times.

**What This Means for Business:**

If we made lending rules based ONLY on edufin_small and the patterns don't hold in edufin_national, we could:
- Reject good customers (false alarm from small data)
- Miss real risks (small data didn't show them)
- Waste money on wrong strategies

**Key Takeaway:** Always validate findings on larger datasets before making major business decisions. Small data is good for exploration and generating ideas, but big data is needed for confirmation. If a pattern disappears at scale, don't trust it - it was probably never real or was specific to that small sample's unique circumstances.

## SUBMISSION READINESS CHECK

**Before submitting, verify:**

✓ Data Validation (Cell 2) passed

✓ All 5 SQL queries (Cells 3-7) executed without errors

✓ Executive Summary (Cell 8) - all 4 questions answered

✓ Investigation Summary (Cell 9) - 3-4 sentence synthesis written

✓ WHY Gate (Cell 10) - all 3 questions answered with business reasoning

✓ Formatting: Currency in Crores/Lakhs, percentages to 2 decimals

✓ Column aliases: Business-friendly names (not raw database column names)

✓ Results reviewed: Do the numbers make business sense?

---

## SUBMISSION PROCESS

**Step 1:** Run Cell 12 below
- Enter your name and select day
- Get your submission filename and form link

**Step 2:** Export this notebook as HTML
1. Go to **File** menu (top left)
2. Click on **Export**
3. Select **IPython notebook** format from the dropdown
4. Click **Export** button
5. Your browser will download the file

**Step 3:** Rename the downloaded file
- Use the exact filename shown in Cell 12 output
- Example: `Name_Batch_no_Day1.ipynb`

**Step 4:** Submit via Google Form
- Open the form link from Cell 12 output
- Upload your renamed ipynb file
- Note your entry number from form confirmation (tracking ID for feedback)

**Step 5:** Wait for feedback
- Expect feedback within 24 hours via WhatsApp

---

**Reminder:** Partial submissions accepted. If only 3 of 5 queries work, submit what you have with documentation of where you got stuck.

In [0]:
import re

FORM_LINK = "https://forms.gle/7xWVjeLRahSEMYXD9"  

dbutils.widgets.removeAll()
dbutils.widgets.text("Name", "Ashish Mishra")
dbutils.widgets.text("EnrollmentID", "SAP174B9")
dbutils.widgets.dropdown("day", "Day2", ["Day1", "Day2", "Day3"])

name = dbutils.widgets.get("Name")
batch = dbutils.widgets.get("EnrollmentID")
day = dbutils.widgets.get("day")

if not name or not batch:
    print("⚠️  Enter your name and BatchNo above, then run this cell")
else:
    clean_name = re.sub(r'[^a-zA-Z0-9]', '', name.lower())
    filename = f"{clean_name}_{batch}_{day}.ipynb"
    
    print("File -> Export -> IPython notebook")
    print(f"Rename to: {filename}")
    print(f"Submit the file in the form link: {FORM_LINK}")
    print("\nPlease await for submission review.")

File -> Export -> IPython notebook
Rename to: ashishmishra_SAP174B9_Day2.ipynb
Submit the file in the form link: https://forms.gle/7xWVjeLRahSEMYXD9

Please await for submission review.
